# Assignment 2: Normalized TF and IDF Variants for Language Identification


1. Two TF normalization schemes
   - `norm_total`: divide raw term count by total number of terms in the sentence
   - `norm_max`: divide raw term count by the count of the most frequent term in the sentence
2. One IDF normalization scheme
   - `norm_idf`: `log(max_df / df_t)`, where `max_df` is the document frequency of the
     most frequent term across the whole corpus (instead of `log(N/df_t)`)
3. All 6 combinations of {raw, norm_total, norm_max} TF x {standard, norm} IDF,
   each trained with the same Logistic Regression classifier and compared on macro-F1.

No sklearn classes are used anywhere.


## 1. Dataset construction (same as Assignment 1)

In [1]:
!pip install datasets scipy numpy tqdm

import re
import math
import numpy as np
import scipy.sparse as sp
from datasets import load_dataset
from collections import defaultdict, Counter
from tqdm.notebook import tqdm


In [2]:
dataset_name = "ai4bharat/IndicCorpV2"

# Same 23 language_script splits used in Assignment 1
languages = [
    'asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'gom_Deva',
    'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'mai_Deva',
    'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya',
    'pan_Guru', 'san_Deva', 'snd_Deva', 'tam_Taml', 'tel_Telu',
    'urd_Arab', 'khasi', 'santhali'
]

train_data, val_data, test_data = [], [], []
train_labels, val_labels, test_labels = [], [], []

split_pattern = re.compile(r'[।!?.\n]+')

def get_sentences(text):
    sentences = split_pattern.split(text)
    return [s.strip() for s in sentences if len(s.strip()) > 15]

print(f"Fetching data for {len(languages)} languages. This will take a few minutes...")

for lang_idx, lang_split in enumerate(tqdm(languages, desc="Processing Languages")):
    dataset = load_dataset(dataset_name, split=lang_split, streaming=True)

    lang_sentences = []
    for sample in dataset:
        sentences = get_sentences(sample['text'])
        for sent in sentences:
            if len(lang_sentences) < 1000:
                lang_sentences.append(sent)
            else:
                break
        if len(lang_sentences) == 1000:
            break

    # Stratified split: 800 train, 100 val, 100 test
    train_data.extend(lang_sentences[:800]);   train_labels.extend([lang_idx] * 800)
    val_data.extend(lang_sentences[800:900]);  val_labels.extend([lang_idx] * 100)
    test_data.extend(lang_sentences[900:1000]);test_labels.extend([lang_idx] * 100)

y_train = np.array(train_labels)
y_val   = np.array(val_labels)
y_test  = np.array(test_labels)

print(f"Total Training Samples: {len(train_data)}")
print(f"Total Validation Samples: {len(val_data)}")
print(f"Total Test Samples: {len(test_data)}")


Fetching data for 23 languages. This will take a few minutes...


Processing Languages:   0%|          | 0/23 [00:00<?, ?it/s]

Total Training Samples: 18400
Total Validation Samples: 2300
Total Test Samples: 2300


## 2. TF-IDF with configurable TF / IDF normalization

Design: build the vocabulary and every term's **document frequency** exactly once
(this is expensive), producing a plain **raw term-count matrix** for train/val/test.
Every TF/IDF *weighting* is then just cheap sparse arithmetic on top of that single
count matrix, instead of re-tokenizing the corpus 6 times.

- Vocabulary / n-gram extraction (word unigrams+bigrams, char 2/3/4-grams) is
  unchanged from Assignment 1.
- `raw` TF: the term count itself, `f(t,d)`.
- `norm_total` TF: `f(t,d) / sum_t' f(t',d)` — normalized by the total number of
  terms (word/char n-grams) in the sentence.
- `norm_max` TF: `f(t,d) / max_t' f(t',d)` — normalized by the frequency of the
  single most frequent term in that sentence.
- `standard` IDF (Assignment 1): `log((N+1)/(df_t+1)) + 1`
- `norm` IDF: `log(max_df / df_t)`, where `max_df = max_t df_t` is the document
  frequency of the most frequent term **across the whole corpus**. The most
  frequent term in the corpus gets an IDF of 0 (like standard IDF, it downweights
  ubiquitous terms), rare terms get the largest weight.


In [3]:
class CustomTFIDFCounter:
    """Builds vocabulary + document frequencies and produces a RAW COUNT matrix.
    All TF/IDF weighting variants are derived from this single matrix afterwards.
    """

    def __init__(self, min_df=5):
        self.min_df = min_df
        self.vocab = {}
        self.doc_freq = {}   # df_t for every term kept in the vocabulary
        self.max_df = None   # document frequency of the most frequent term (for norm IDF)
        self.N = None

    def extract_ngrams(self, text):
        words = text.split()
        ngrams = []

        # Word Unigrams
        ngrams.extend(words)
        # Word Bigrams
        ngrams.extend([f"{words[i]}_{words[i+1]}" for i in range(len(words) - 1)])

        # Character 2, 3, 4-grams
        text_no_space = text.replace(" ", "")
        for n in [2, 3, 4]:
            ngrams.extend([text_no_space[i:i + n] for i in range(len(text_no_space) - n + 1)])

        return ngrams

    def fit(self, documents):
        print("Building vocabulary and document frequencies...")
        doc_freq = Counter()

        for doc in tqdm(documents, desc="Fitting vocabulary"):
            unique_ngrams = set(self.extract_ngrams(doc))
            for ngram in unique_ngrams:
                doc_freq[ngram] += 1

        vocab_idx = 0
        self.N = len(documents)
        for ngram, freq in doc_freq.items():
            if freq >= self.min_df:
                self.vocab[ngram] = vocab_idx
                self.doc_freq[ngram] = freq
                vocab_idx += 1

        self.max_df = max(self.doc_freq.values())
        print(f"Vocabulary size: {len(self.vocab)}")
        print(f"Most frequent term appears in {self.max_df} / {self.N} documents")

    def transform_counts(self, documents):
        """Returns the RAW term-count sparse matrix (no TF/IDF weighting applied)."""
        rows, cols, data = [], [], []

        for row_idx, doc in enumerate(tqdm(documents, desc="Counting terms")):
            term_freq = Counter(self.extract_ngrams(doc))
            for ngram, count in term_freq.items():
                if ngram in self.vocab:
                    rows.append(row_idx)
                    cols.append(self.vocab[ngram])
                    data.append(float(count))

        return sp.csr_matrix((data, (rows, cols)), shape=(len(documents), len(self.vocab)))

    # ---- IDF vectors ----
    def idf_standard(self):
        """log((N+1)/(df_t+1)) + 1, same formula as Assignment 1."""
        idf = np.zeros(len(self.vocab))
        for ngram, idx in self.vocab.items():
            df = self.doc_freq[ngram]
            idf[idx] = math.log((self.N + 1) / (df + 1)) + 1
        return idf

    def idf_normalized(self):
        """log(max_df / df_t) -- normalized by the DF of the most frequent term
        across the corpus. The most frequent term gets idf = 0."""
        idf = np.zeros(len(self.vocab))
        for ngram, idx in self.vocab.items():
            df = self.doc_freq[ngram]
            idf[idx] = math.log(self.max_df / df)
        return idf


In [4]:
# Fit vocabulary once on the training data, then get raw count matrices
vectorizer = CustomTFIDFCounter(min_df=5)
vectorizer.fit(train_data)

counts_train = vectorizer.transform_counts(train_data)
counts_val   = vectorizer.transform_counts(val_data)
counts_test  = vectorizer.transform_counts(test_data)


Building vocabulary and document frequencies...


Fitting vocabulary:   0%|          | 0/18400 [00:00<?, ?it/s]

Vocabulary size: 132377
Most frequent term appears in 2712 / 18400 documents


Counting terms:   0%|          | 0/18400 [00:00<?, ?it/s]

Counting terms:   0%|          | 0/2300 [00:00<?, ?it/s]

Counting terms:   0%|          | 0/2300 [00:00<?, ?it/s]

## 3. TF and IDF weighting variants (built on top of the raw count matrices)

In [5]:
def sparse_row_sum(csr_mat):
    """Total term count per row (i.e. total number of n-grams in the sentence)."""
    row_sum = np.array(csr_mat.sum(axis=1)).flatten()
    row_sum[row_sum == 0] = 1.0  # avoid div-by-zero for empty rows
    return row_sum

def sparse_row_max(csr_mat):
    """Count of the single most frequent term per row."""
    data, indptr = csr_mat.data, csr_mat.indptr
    n_rows = csr_mat.shape[0]
    row_max = np.zeros(n_rows)
    for i in range(n_rows):
        start, end = indptr[i], indptr[i + 1]
        if end > start:
            row_max[i] = data[start:end].max()
    row_max[row_max == 0] = 1.0
    return row_max


def apply_tf(counts_matrix, tf_mode):
    """tf_mode in {'raw', 'norm_total', 'norm_max'}"""
    if tf_mode == 'raw':
        return counts_matrix.copy()
    elif tf_mode == 'norm_total':
        row_sum = sparse_row_sum(counts_matrix)
        return counts_matrix.multiply(1.0 / row_sum[:, None]).tocsr()
    elif tf_mode == 'norm_max':
        row_max = sparse_row_max(counts_matrix)
        return counts_matrix.multiply(1.0 / row_max[:, None]).tocsr()
    else:
        raise ValueError(tf_mode)


def apply_idf(tf_matrix, idf_vector):
    weighted = tf_matrix.multiply(idf_vector[None, :]).tocsr()
    # L2-normalize each row (kept consistent across all 6 combinations for a fair comparison)
    norms = np.sqrt(np.array(weighted.multiply(weighted).sum(axis=1)).flatten())
    norms[norms == 0] = 1.0
    return weighted.multiply(1.0 / norms[:, None]).tocsr()


idf_standard_vec   = vectorizer.idf_standard()
idf_normalized_vec = vectorizer.idf_normalized()

idf_variants = {
    'standard_idf': idf_standard_vec,
    'norm_idf':      idf_normalized_vec,
}
tf_variants = ['raw', 'norm_total', 'norm_max']

# Build all 6 (TF x IDF) feature matrices for train / val / test
features = {}  # (tf_mode, idf_mode) -> (X_train, X_val, X_test)

for tf_mode in tf_variants:
    tf_train = apply_tf(counts_train, tf_mode)
    tf_val   = apply_tf(counts_val, tf_mode)
    tf_test  = apply_tf(counts_test, tf_mode)

    for idf_name, idf_vec in idf_variants.items():
        X_train = apply_idf(tf_train, idf_vec)
        X_val   = apply_idf(tf_val, idf_vec)
        X_test  = apply_idf(tf_test, idf_vec)
        features[(tf_mode, idf_name)] = (X_train, X_val, X_test)
        print(f"Built features for TF='{tf_mode}', IDF='{idf_name}'  "
              f"(train shape={X_train.shape})")


Built features for TF='raw', IDF='standard_idf'  (train shape=(18400, 132377))
Built features for TF='raw', IDF='norm_idf'  (train shape=(18400, 132377))
Built features for TF='norm_total', IDF='standard_idf'  (train shape=(18400, 132377))
Built features for TF='norm_total', IDF='norm_idf'  (train shape=(18400, 132377))
Built features for TF='norm_max', IDF='standard_idf'  (train shape=(18400, 132377))
Built features for TF='norm_max', IDF='norm_idf'  (train shape=(18400, 132377))


## 4. Custom Logistic Regression (same as Assignment 1)

In [6]:
class CustomLogisticRegression:
    def __init__(self, learning_rate=0.5, epochs=50, batch_size=512, num_classes=23):
        self.lr = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.num_classes = num_classes
        self.weights = None
        self.bias = None

    def softmax(self, z):
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def fit(self, X, y, verbose=False):
        num_samples, num_features = X.shape
        self.weights = np.zeros((num_features, self.num_classes))
        self.bias = np.zeros(self.num_classes)

        y_one_hot = np.zeros((num_samples, self.num_classes))
        y_one_hot[np.arange(num_samples), y] = 1

        for epoch in range(self.epochs):
            indices = np.arange(num_samples)
            np.random.shuffle(indices)
            epoch_loss = 0
            n_batches = 0

            for i in range(0, num_samples, self.batch_size):
                batch_indices = indices[i:i + self.batch_size]
                X_batch = X[batch_indices]
                y_batch = y_one_hot[batch_indices]

                linear_model = X_batch.dot(self.weights) + self.bias
                predictions = self.softmax(linear_model)

                loss = -np.mean(np.sum(y_batch * np.log(predictions + 1e-9), axis=1))
                epoch_loss += loss
                n_batches += 1

                dz = predictions - y_batch
                dw = (X_batch.T.dot(dz)) / self.batch_size
                db = np.sum(dz, axis=0) / self.batch_size

                self.weights -= self.lr * dw
                self.bias -= self.lr * db

            if verbose and (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch + 1}/{self.epochs} - Loss: {epoch_loss / n_batches:.4f}")

    def predict(self, X):
        linear_model = X.dot(self.weights) + self.bias
        predictions = self.softmax(linear_model)
        return np.argmax(predictions, axis=1)


## 5. Custom macro-F1 (same as Assignment 1)

In [7]:
def macro_f1_score(y_true, y_pred, num_classes):
    f1_scores = []
    for c in range(num_classes):
        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        f1_scores.append(f1)

    return np.mean(f1_scores)


## 6. Train and evaluate all 6 TF/IDF combinations

Same hyperparameters used for every run so the comparison isolates the effect of the
TF/IDF weighting scheme.


In [8]:
results = []

for (tf_mode, idf_name), (X_train, X_val, X_test) in features.items():
    print(f"\n=== Training: TF='{tf_mode}', IDF='{idf_name}' ===")
    model = CustomLogisticRegression(
        learning_rate=0.5, epochs=50, batch_size=512, num_classes=len(languages)
    )
    model.fit(X_train, y_train, verbose=True)

    val_preds  = model.predict(X_val)
    test_preds = model.predict(X_test)

    val_f1  = macro_f1_score(y_val, val_preds, len(languages))
    test_f1 = macro_f1_score(y_test, test_preds, len(languages))

    print(f"Validation Macro-F1: {val_f1:.4f}  |  Test Macro-F1: {test_f1:.4f}")

    results.append({
        'TF': tf_mode,
        'IDF': idf_name,
        'Val Macro-F1': val_f1,
        'Test Macro-F1': test_f1,
    })



=== Training: TF='raw', IDF='standard_idf' ===
  Epoch 10/50 - Loss: 2.6659
  Epoch 20/50 - Loss: 2.2374
  Epoch 30/50 - Loss: 1.8939
  Epoch 40/50 - Loss: 1.6281
  Epoch 50/50 - Loss: 1.4237
Validation Macro-F1: 0.9158  |  Test Macro-F1: 0.9255

=== Training: TF='raw', IDF='norm_idf' ===
  Epoch 10/50 - Loss: 2.8419
  Epoch 20/50 - Loss: 2.5542
  Epoch 30/50 - Loss: 2.2967
  Epoch 40/50 - Loss: 2.0717
  Epoch 50/50 - Loss: 1.8782
Validation Macro-F1: 0.9285  |  Test Macro-F1: 0.9408

=== Training: TF='norm_total', IDF='standard_idf' ===
  Epoch 10/50 - Loss: 2.6659
  Epoch 20/50 - Loss: 2.2373
  Epoch 30/50 - Loss: 1.8938
  Epoch 40/50 - Loss: 1.6282
  Epoch 50/50 - Loss: 1.4237
Validation Macro-F1: 0.9153  |  Test Macro-F1: 0.9236

=== Training: TF='norm_total', IDF='norm_idf' ===
  Epoch 10/50 - Loss: 2.8419
  Epoch 20/50 - Loss: 2.5542
  Epoch 30/50 - Loss: 2.2967
  Epoch 40/50 - Loss: 2.0717
  Epoch 50/50 - Loss: 1.8780
Validation Macro-F1: 0.9286  |  Test Macro-F1: 0.9419

=== T

## 7. Comparison table

In [9]:
print(f"{'TF Mode':<14}{'IDF Mode':<16}{'Val Macro-F1':<16}{'Test Macro-F1':<16}")
print("-" * 62)
for r in sorted(results, key=lambda x: -x['Test Macro-F1']):
    print(f"{r['TF']:<14}{r['IDF']:<16}{r['Val Macro-F1']:<16.4f}{r['Test Macro-F1']:<16.4f}")


TF Mode       IDF Mode        Val Macro-F1    Test Macro-F1   
--------------------------------------------------------------
norm_total    norm_idf        0.9286          0.9419          
raw           norm_idf        0.9285          0.9408          
norm_max      norm_idf        0.9250          0.9404          
raw           standard_idf    0.9158          0.9255          
norm_total    standard_idf    0.9153          0.9236          
norm_max      standard_idf    0.9146          0.9235          
